# Upload tiles to OMERO

This notebook uploads a set of brightfield tiles to omero and parses the necessary metadata.

In [13]:
import omero
import os
import glob
from pathlib import Path
from skimage import io
import numpy as np
from bioio import BioImage
import ezomero
from omero.model.enums import UnitsLength
from ome_zarr import OMEZarrScene
import itertools
import tqdm
from ultralytics import YOLO
import tifffile as tff

#from functions import load_pos_file, get_tile_metadata

Log into omero:

In [18]:
host = 'omero-nfdi.uni-muenster.de'
port = 4064
user = 'JohannesSoltwedel'
group = None
secure=True
password = "BYMCfyZpMDLC"

conn = ezomero.connect(host=host, user=user, secure=secure, port=port, group=group, password=password)

## Upload a complete, stitched overview image in tiles

In [3]:
scene = OMEZarrScene.from_ome_zarr(r'Z:\Collaborations\aif-botton-collaboration\Johannes\20260811\scene.ome.zarr')

In [4]:
dataset_id = 552
channel_names = {1: 'Brightfield'}

In [ ]:
for name, img_ms in scene.images.items():
    shape = img_ms.images[0].data.shape
    data = img_ms.images[0].data.rechunk((shape[0], 2048, 2048))

    unit_x = omero.model.LengthI(img_ms.images[0].scale["x"], UnitsLength.MICROMETER)
    unit_y = omero.model.LengthI(img_ms.images[0].scale["y"], UnitsLength.MICROMETER)

    ids = []
    for idx in itertools.product(*map(range, data.blocks.shape)):
        tile_mip = data.blocks[idx].min(axis=0)

        image_id = ezomero.post_image(conn, image=tile_mip.compute()[None, None, None], dataset_id=dataset_id,
                                      image_name=f"{img_ms.name}_{idx}", dim_order='tczyx',
                                      across_groups=False)
        
        image_object = conn.getObject("Image", image_id)
        primary_pixels_object = image_object.getPrimaryPixels()._obj
        primary_pixels_object.setPhysicalSizeX(unit_x)
        primary_pixels_object.setPhysicalSizeY(unit_y)
        conn.getUpdateService().saveObject(primary_pixels_object)

        ids.append(image_id)
    
        conn.setChannelNames(data_type='Image', ids=[image_id], nameDict=channel_names)

## Dataset for focus detector

If there already is an object detector model, we can use it to find the objects and crop out stacks to train our focus detector.

In [26]:
root = r"Z:\Collaborations\aif-botton-collaboration\Johannes\20260826\2026026_BF_W2_seelcted_1"
dataset_id = 553

img_files = glob.glob(os.path.join(root, "*.tif"))
unit_x = omero.model.LengthI(0.325, UnitsLength.MICROMETER)
unit_y = omero.model.LengthI(0.325, UnitsLength.MICROMETER)
unit_z = omero.model.LengthI(10, UnitsLength.MICROMETER)

for f in img_files:
    data = tff.imread(f, is_ome=False, is_mmstack=False)
    image_id = ezomero.post_image(conn, image=data[None, None, :], dataset_id=dataset_id,
                                          image_name=Path(f).stem, dim_order='tczyx',
                                          across_groups=False)
    image_object = conn.getObject("Image", image_id)
    primary_pixels_object = image_object.getPrimaryPixels()._obj
    primary_pixels_object.setPhysicalSizeX(unit_x)
    primary_pixels_object.setPhysicalSizeY(unit_y)
    primary_pixels_object.setPhysicalSizeZ(unit_z)
    conn.getUpdateService().saveObject(primary_pixels_object)

ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @71886> invalid valueoffset=0')
ERROR:tifffile:<TiffTag.fromfile> raised TiffFileError('<TiffTag 50839 @7

In [23]:
primary_pixels_object.setPhysicalSizeZ(10)